# Forecasting & Strategic Scenario Modeling — SPICE (Group 7)\n## CMPT 3835 — Machine Learning in the Workplace II\n\n**Team:** Sandesh Khanal, Farhan ur Rahman, Alma Soria\n\n**Objective:** Predict short-term production and revenue under uncertainty.\n\n**Key Question:** *What production and revenue outcomes can stakeholders expect in the near future?*\n\n**Focus Areas:**\n1. 30-day production forecast\n2. Seasonal decomposition\n3. Confidence intervals\n4. Best/worst-case revenue scenarios (using AESO prices)\n\n**External Data:** Weather context + AESO electricity prices\n\n**Business Lens:** Forward planning & decision support for SPICE stakeholders\n\n---\n> **Google Colab version** — Upload the Bissell Excel file when prompted below.

---\n# Section 1: Data Loading & Preparation\nUpload the Bissell Thrift Store production Excel file when prompted below.\n\n**File needed:** `Bissell_Thrift_118_Ave_01012025-01012026.xlsx`

In [ ]:
# --- Install dependencies (Colab may not have statsmodels pre-installed) ---
!pip install statsmodels openpyxl --quiet

In [ ]:
# --- Imports ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.figsize": (14, 5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

# --- SPICE Brand Colors ---
SPICE_YELLOW = "#F2A900"
SPICE_DARK   = "#1B2A4A"
SPICE_GREEN  = "#4CAF50"
SPICE_ORANGE = "#FF6B35"

print("Libraries loaded successfully.")

In [ ]:
# --- Upload Bissell Excel file (Colab file upload) ---
from google.colab import files
import io

print("Please upload: Bissell_Thrift_118_Ave_01012025-01012026.xlsx")
uploaded = files.upload()

# Get the uploaded file
filename = list(uploaded.keys())[0]
print(f"\nUploaded: {filename}")

raw = pd.read_excel(io.BytesIO(uploaded[filename]), sheet_name="Production 2025", header=0, skiprows=[1])
raw.columns = [
    "date", "inv1_kwh", "inv2_kwh", "inv3_kwh",
    "inv1_kwh_per_kwp", "inv2_kwh_per_kwp", "inv3_kwh_per_kwp",
    "total_kwh",
]
raw["date"] = pd.to_datetime(raw["date"], format="%d.%m.%Y", errors="coerce")
raw = raw.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)

# Ensure numeric
for col in raw.columns[1:]:
    raw[col] = pd.to_numeric(raw[col], errors="coerce").fillna(0)

print(f"\nLoaded {len(raw)} days: {raw['date'].min().date()} to {raw['date'].max().date()}")
print(f"Total production: {raw['total_kwh'].sum():,.1f} kWh")
print(f"Non-zero production days: {(raw['total_kwh'] > 0).sum()}")
raw.head()

In [ ]:
# --- Quick overview of the production time series ---
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Daily production
axes[0].bar(raw["date"], raw["total_kwh"], color=SPICE_YELLOW, edgecolor="white", linewidth=0.3)
axes[0].set_title("Bissell Thrift — Daily Solar Production", fontweight="bold", color=SPICE_DARK)
axes[0].set_ylabel("kWh")
axes[0].xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
axes[0].tick_params(axis="x", rotation=45)

# Monthly aggregation
monthly = raw.groupby(raw["date"].dt.to_period("M"))["total_kwh"].sum().reset_index()
monthly["date"] = monthly["date"].dt.to_timestamp()
axes[1].bar(monthly["date"], monthly["total_kwh"], width=20, color=SPICE_DARK, edgecolor="white")
axes[1].set_title("Monthly Production Summary", fontweight="bold", color=SPICE_DARK)
axes[1].set_ylabel("kWh")
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

print(f"\nMonthly breakdown:")
for _, row in monthly.iterrows():
    print(f"  {row['date'].strftime('%B %Y'):>15s}: {row['total_kwh']:>8.1f} kWh")

---\n# Section 2: External Data — AESO Electricity Prices & Weather Context\n\n**AESO (Alberta Electric System Operator)** publishes wholesale pool prices for Alberta's electricity market.\nWe use published AESO monthly average pool prices for the Bissell data period (Aug–Dec 2025) to build revenue scenarios.\n\n**Weather context:** Edmonton experiences significant seasonal variation in daylight hours and solar irradiance,\nwhich directly explains the production decline from summer to winter observed in the Bissell data.

In [ ]:
# --- External Data: AESO Pool Prices (Alberta wholesale electricity) ---
# Source: AESO published monthly average pool prices for Alberta (2025)
# https://www.aeso.ca/market/market-and-system-reporting/
#
# These are approximate monthly averages from AESO public reporting.
# In practice, SPICE's revenue depends on their specific contract (PPA or
# micro-generation credit), but pool prices set the market context.

aeso_prices = pd.DataFrame({
    "month": pd.to_datetime(["2025-08-01", "2025-09-01", "2025-10-01", "2025-11-01", "2025-12-01"]),
    "pool_price_mwh": [75.0, 85.0, 105.0, 130.0, 145.0],   # $/MWh avg
    "pool_price_kwh": [0.075, 0.085, 0.105, 0.130, 0.145],  # $/kWh
})

# Reference rates for scenarios
REGULATED_RATE = 0.18   # $/kWh — Alberta RRO blended average
MICRO_GEN_CREDIT = 0.12 # $/kWh — typical micro-generation credit rate

print("AESO Monthly Average Pool Prices (2025):")
print("=" * 55)
for _, r in aeso_prices.iterrows():
    print(f"  {r['month'].strftime('%B %Y'):>15s}: ${r['pool_price_mwh']:>6.1f}/MWh  (${r['pool_price_kwh']:.3f}/kWh)")

print(f"\nReference rates for revenue scenarios:")
print(f"  Regulated Rate (RRO):     ${REGULATED_RATE:.2f}/kWh")
print(f"  Micro-generation credit:  ${MICRO_GEN_CREDIT:.2f}/kWh")
print(f"  AESO pool (period avg):   ${aeso_prices['pool_price_kwh'].mean():.3f}/kWh")

In [ ]:
# --- External Data: Edmonton Weather Context ---
# Edmonton solar resource varies dramatically by season:
#   - Summer solstice (Jun 21): ~17h daylight, high solar altitude
#   - Winter solstice (Dec 21): ~7.5h daylight, low solar altitude
#
# Source: Natural Resources Canada, Edmonton solar resource data

edmonton_solar = pd.DataFrame({
    "month": ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
              "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"],
    "avg_daylight_hrs": [8.1, 9.8, 11.8, 13.9, 15.7, 17.0,
                         16.5, 14.8, 12.7, 10.6, 8.7, 7.5],
    "avg_solar_irradiance_kwh_m2_day": [1.2, 2.1, 3.4, 4.6, 5.5, 5.8,
                                         5.6, 4.7, 3.3, 2.1, 1.2, 0.9],
})

fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()

x = range(12)
ax1.bar(x, edmonton_solar["avg_solar_irradiance_kwh_m2_day"], color=SPICE_YELLOW,
        alpha=0.8, label="Solar Irradiance (kWh/m²/day)")
ax2.plot(x, edmonton_solar["avg_daylight_hrs"], color=SPICE_DARK,
         marker="o", linewidth=2.5, label="Daylight Hours")

ax1.set_xticks(x)
ax1.set_xticklabels(edmonton_solar["month"])
ax1.set_ylabel("Avg Solar Irradiance (kWh/m²/day)", color=SPICE_YELLOW)
ax2.set_ylabel("Daylight Hours", color=SPICE_DARK)
ax1.set_title("Edmonton Seasonal Solar Resource — Why Production Drops in Winter",
              fontweight="bold", color=SPICE_DARK)

# Highlight Bissell data period
for i in [7, 8, 9, 10, 11]:  # Aug-Dec
    ax1.axvspan(i - 0.4, i + 0.4, alpha=0.08, color=SPICE_ORANGE)
ax1.text(9.5, 5.5, "Bissell data\nperiod", ha="center", fontsize=9,
         style="italic", color=SPICE_ORANGE)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

plt.tight_layout()
plt.show()

print("\nKey insight: Bissell data spans Aug-Dec — the steepest decline in solar resource.")
print("This seasonal pattern is the primary driver of the production trend we observe.")

---\n# Section 3: Seasonal Decomposition\n\nDecompose the Bissell daily production into **trend**, **seasonal** (weekly cycle), and **residual** components.\nThis reveals the underlying patterns driving production changes.

In [ ]:
# --- Seasonal Decomposition ---
# Use only non-zero production days for meaningful decomposition.
# We set the time series index to daily frequency and fill gaps.

ts = raw[["date", "total_kwh"]].copy()
ts = ts.set_index("date").asfreq("D")
ts["total_kwh"] = ts["total_kwh"].fillna(0)

# Use a 7-day period (weekly cycle) for decomposition
# Only decompose the period with meaningful production (Aug 20 - Nov 30)
ts_active = ts.loc["2025-08-20":"2025-11-30"]

decomp = seasonal_decompose(ts_active["total_kwh"], model="additive", period=7)

fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

axes[0].plot(decomp.observed, color=SPICE_DARK, linewidth=1.2)
axes[0].set_ylabel("Observed\n(kWh)")
axes[0].set_title("Seasonal Decomposition of Bissell Daily Production (7-day cycle)",
                   fontweight="bold", color=SPICE_DARK, fontsize=13)

axes[1].plot(decomp.trend, color=SPICE_ORANGE, linewidth=2)
axes[1].set_ylabel("Trend\n(kWh)")
axes[1].fill_between(decomp.trend.index, decomp.trend, alpha=0.2, color=SPICE_ORANGE)

axes[2].plot(decomp.seasonal, color=SPICE_GREEN, linewidth=1.2)
axes[2].set_ylabel("Seasonal\n(kWh)")
axes[2].axhline(0, color="gray", linestyle="--", linewidth=0.5)

axes[3].scatter(decomp.resid.index, decomp.resid, color=SPICE_YELLOW, s=15, alpha=0.7)
axes[3].axhline(0, color="gray", linestyle="--", linewidth=0.5)
axes[3].set_ylabel("Residual\n(kWh)")
axes[3].set_xlabel("Date")

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))

plt.tight_layout()
plt.show()

print("Decomposition findings:")
print(f"  Trend: Clear downward slope from ~{decomp.trend.dropna().iloc[:7].mean():.0f} kWh/day (Aug)")
print(f"         to ~{decomp.trend.dropna().iloc[-7:].mean():.0f} kWh/day (Nov)")
print(f"  Seasonal (weekly): amplitude ~{decomp.seasonal.max() - decomp.seasonal.min():.1f} kWh")
print(f"  Residual std: {decomp.resid.dropna().std():.1f} kWh — represents day-to-day weather variability")

---\n# Section 4: 30-Day Production Forecast with Confidence Intervals\n\nWe use **Holt-Winters Exponential Smoothing** (triple exponential smoothing) to forecast the next 30 days of production.\nThis method captures both the downward trend and any weekly seasonality.\n\nWe also compute **confidence intervals** using the model's forecast variance to show the range of uncertainty.

In [ ]:
# --- 30-Day Production Forecast using Holt-Winters ---

# Prepare time series: use the active production period (Aug 20 - Nov 30)
ts_model = ts_active["total_kwh"].copy()

# Fit Holt-Winters with additive trend and additive seasonality (7-day cycle)
hw_model = ExponentialSmoothing(
    ts_model,
    trend="add",
    seasonal="add",
    seasonal_periods=7,
    damped_trend=True,
).fit(optimized=True)

# Forecast next 30 days (Dec 1 - Dec 30, 2025)
forecast_steps = 30
forecast = hw_model.forecast(forecast_steps)
forecast.index = pd.date_range(start="2025-12-01", periods=forecast_steps, freq="D")

# Clamp negative forecasts to zero (solar can't produce negative energy)
forecast = forecast.clip(lower=0)

# --- Confidence Intervals ---
# Use residual standard deviation to build prediction intervals
residuals = hw_model.resid.dropna()
resid_std = residuals.std()

# 80% and 95% confidence intervals
z_80 = 1.28
z_95 = 1.96

ci_80_lower = (forecast - z_80 * resid_std).clip(lower=0)
ci_80_upper = forecast + z_80 * resid_std
ci_95_lower = (forecast - z_95 * resid_std).clip(lower=0)
ci_95_upper = forecast + z_95 * resid_std

print(f"Model: Holt-Winters (damped additive trend + 7-day seasonal)")
print(f"Residual std: {resid_std:.2f} kWh")
print(f"Forecast period: {forecast.index[0].date()} to {forecast.index[-1].date()}")
print(f"Forecasted total (30 days): {forecast.sum():.1f} kWh")
print(f"  95% CI range: [{ci_95_lower.sum():.1f}, {ci_95_upper.sum():.1f}] kWh")

In [ ]:
# --- Visualization: 30-Day Forecast with Confidence Intervals ---

fig, ax = plt.subplots(figsize=(16, 7))

# Historical data
ax.plot(ts_model.index, ts_model.values, color=SPICE_DARK, linewidth=1.2, label="Observed (Aug-Nov)")

# Actual December data for comparison
ts_dec = ts.loc["2025-12-01":"2025-12-31"]
ax.scatter(ts_dec.index, ts_dec["total_kwh"], color=SPICE_GREEN, s=30, zorder=5,
           label="Actual December (ground truth)")

# Forecast
ax.plot(forecast.index, forecast.values, color=SPICE_ORANGE, linewidth=2.5,
        linestyle="--", label="30-Day Forecast (Holt-Winters)")

# Confidence intervals
ax.fill_between(forecast.index, ci_80_lower, ci_80_upper,
                alpha=0.3, color=SPICE_YELLOW, label="80% Confidence Interval")
ax.fill_between(forecast.index, ci_95_lower, ci_95_upper,
                alpha=0.15, color=SPICE_ORANGE, label="95% Confidence Interval")

# Vertical line at forecast start
ax.axvline(pd.Timestamp("2025-12-01"), color="gray", linestyle=":", linewidth=1)
ax.text(pd.Timestamp("2025-12-01"), ax.get_ylim()[1] * 0.9, " Forecast\n starts",
        fontsize=9, color="gray")

ax.set_title("Bissell Thrift Store — 30-Day Production Forecast with Confidence Intervals",
             fontweight="bold", color=SPICE_DARK, fontsize=14)
ax.set_ylabel("Daily Production (kWh)")
ax.set_xlabel("Date")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
ax.legend(loc="upper right", fontsize=10)
ax.set_ylim(bottom=-5)

plt.tight_layout()
plt.show()

print("\nForecast vs Actual (December):")
print(f"  Forecasted Dec total:  {forecast.sum():>8.1f} kWh")
actual_dec = ts_dec["total_kwh"].sum()
print(f"  Actual Dec total:      {actual_dec:>8.1f} kWh")
if actual_dec > 0:
    print(f"  Forecast error:        {abs(forecast.sum() - actual_dec):>8.1f} kWh")

---\n# Section 5: Best/Worst-Case Revenue Scenarios\n\nWe combine the production forecast with three pricing models to estimate revenue under different scenarios:\n\n| Scenario | Price Assumption | Rationale |\n|---|---|---|\n| **Best Case** | $0.18/kWh (RRO) | Full regulated rate offset — highest value per kWh |\n| **Base Case** | $0.12/kWh (Micro-gen credit) | Typical micro-generation credit paid to small producers |\n| **Worst Case** | AESO pool price (~$0.108/kWh avg) | Wholesale market price — lowest, most volatile |\n\nEach scenario is applied to the **forecasted production** and its **confidence intervals** to show the full range of possible revenue outcomes.

In [ ]:
# --- Revenue Scenario Modeling ---

# Define pricing scenarios
scenarios = {
    "Best Case (RRO $0.18/kWh)": REGULATED_RATE,
    "Base Case (Micro-gen $0.12/kWh)": MICRO_GEN_CREDIT,
    "Worst Case (AESO Pool ~$0.108/kWh)": aeso_prices["pool_price_kwh"].mean(),
}

# Calculate revenue for each scenario across forecast + confidence intervals
print("=" * 70)
print("30-DAY REVENUE FORECAST SCENARIOS (Dec 2025)")
print("=" * 70)
print(f"{'Scenario':<35s} {'Expected':>10s} {'Low (95%)':>12s} {'High (95%)':>12s}")
print("-" * 70)

scenario_results = []
for name, rate in scenarios.items():
    rev_expected = forecast.sum() * rate
    rev_low = ci_95_lower.sum() * rate
    rev_high = ci_95_upper.sum() * rate
    print(f"{name:<35s} ${rev_expected:>8.2f}  ${rev_low:>10.2f}  ${rev_high:>10.2f}")
    scenario_results.append({
        "scenario": name,
        "rate": rate,
        "revenue_expected": rev_expected,
        "revenue_low_95": rev_low,
        "revenue_high_95": rev_high,
        "production_kwh": forecast.sum(),
    })

print("-" * 70)
print(f"\nForecasted production: {forecast.sum():.1f} kWh")
print(f"95% CI production range: [{ci_95_lower.sum():.1f}, {ci_95_upper.sum():.1f}] kWh")

scenario_df = pd.DataFrame(scenario_results)

In [ ]:
# --- Visualization: Revenue Scenarios ---

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Bar chart comparing scenarios
colors_scenario = [SPICE_GREEN, SPICE_YELLOW, SPICE_ORANGE]
bars = axes[0].barh(scenario_df["scenario"], scenario_df["revenue_expected"],
                     color=colors_scenario, edgecolor="white", height=0.5)

# Add error bars for 95% CI
for i, row in scenario_df.iterrows():
    axes[0].plot([row["revenue_low_95"], row["revenue_high_95"]],
                 [i, i], color=SPICE_DARK, linewidth=2, marker="|", markersize=12)

axes[0].set_xlabel("Estimated Revenue (CAD)")
axes[0].set_title("30-Day Revenue Scenarios (Dec 2025)\nwith 95% Confidence Intervals",
                   fontweight="bold", color=SPICE_DARK)

# Add value labels
for i, row in scenario_df.iterrows():
    axes[0].text(row["revenue_expected"] + 0.5, i,
                 f"${row['revenue_expected']:.2f}", va="center", fontweight="bold", fontsize=10)

# Right: Cumulative daily revenue under each scenario
for (name, rate), color in zip(scenarios.items(), colors_scenario):
    cum_revenue = (forecast * rate).cumsum()
    axes[1].plot(forecast.index, cum_revenue, color=color, linewidth=2.5, label=name)
    # Add 95% CI band for best case only (to avoid clutter)
    if "Best" in name:
        axes[1].fill_between(forecast.index,
                             (ci_95_lower * rate).cumsum(),
                             (ci_95_upper * rate).cumsum(),
                             alpha=0.15, color=color)

axes[1].set_title("Cumulative Revenue Over 30-Day Forecast",
                   fontweight="bold", color=SPICE_DARK)
axes[1].set_ylabel("Cumulative Revenue (CAD)")
axes[1].set_xlabel("Date")
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

---\n# Section 6: Annual Projection & Forward Planning\n\nExtend the analysis to a full-year view using Edmonton's known seasonal solar profile.\nThis provides SPICE stakeholders with a **12-month production and revenue outlook** for the Bissell site.

In [ ]:
# --- Annual Production & Revenue Projection ---
# Use Edmonton's monthly solar irradiance profile to project Bissell's annual output.
# We calibrate using actual Aug-Nov data, then scale to other months.

BISSELL_CAPACITY_KWP = 30.0  # 30 kWp system

# Actual monthly production from Bissell data
actual_monthly = {
    8: 1631.98,   # August (partial — 12 days)
    9: 2148.97,   # September
    10: 1784.19,  # October
    11: 662.24,   # November
}

# Full-month August estimate (scale partial to 31 days)
aug_full_est = actual_monthly[8] / 12 * 31  # ~4,215 kWh

# Use Sep-Nov as calibration months (full months)
# Calculate a performance ratio: actual kWh / (irradiance × capacity × days)
irradiance_by_month = {
    1: 1.2, 2: 2.1, 3: 3.4, 4: 4.6, 5: 5.5, 6: 5.8,
    7: 5.6, 8: 4.7, 9: 3.3, 10: 2.1, 11: 1.2, 12: 0.9,
}
days_in_month = {1:31, 2:28, 3:31, 4:30, 5:31, 6:30, 7:31, 8:31, 9:30, 10:31, 11:30, 12:31}

# Calculate performance ratio from Sep-Nov (full months with real data)
pr_values = []
for m in [9, 10, 11]:
    theoretical = irradiance_by_month[m] * BISSELL_CAPACITY_KWP * days_in_month[m] * 0.15  # ~15% panel efficiency
    pr = actual_monthly[m] / theoretical if theoretical > 0 else 0
    pr_values.append(pr)

avg_pr = np.mean(pr_values)

# Project each month
annual_projection = []
for m in range(1, 13):
    theoretical = irradiance_by_month[m] * BISSELL_CAPACITY_KWP * days_in_month[m] * 0.15
    projected = theoretical * avg_pr
    source = "Actual" if m in [9, 10, 11] else "Projected"
    kwh = actual_monthly.get(m, projected) if m in actual_monthly else projected
    annual_projection.append({
        "month": m,
        "month_name": pd.Timestamp(f"2025-{m:02d}-01").strftime("%B"),
        "production_kwh": kwh if m not in [8] else aug_full_est,
        "source": source if m not in [8] else "Estimated (partial data)",
    })

proj_df = pd.DataFrame(annual_projection)
annual_total = proj_df["production_kwh"].sum()

print("Bissell Thrift — Projected Annual Production (30 kWp)")
print("=" * 60)
for _, r in proj_df.iterrows():
    marker = "*" if r["source"] != "Projected" else " "
    print(f" {marker} {r['month_name']:>12s}: {r['production_kwh']:>8.1f} kWh  ({r['source']})")
print("-" * 60)
print(f"   Annual Total: {annual_total:,.1f} kWh")
print(f"\n* = Based on actual Fronius data")

In [ ]:
# --- Annual Revenue Scenarios ---

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Monthly production bar chart
bar_colors = [SPICE_GREEN if r["source"] == "Actual" else
              SPICE_YELLOW if "Estimated" in r["source"] else SPICE_DARK
              for _, r in proj_df.iterrows()]

axes[0].bar(proj_df["month_name"], proj_df["production_kwh"], color=bar_colors, edgecolor="white")
axes[0].set_title("Projected Monthly Production — Bissell 30 kWp",
                   fontweight="bold", color=SPICE_DARK)
axes[0].set_ylabel("kWh")
axes[0].tick_params(axis="x", rotation=45)

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=SPICE_GREEN, label="Actual (Fronius data)"),
    Patch(facecolor=SPICE_YELLOW, label="Estimated (partial data)"),
    Patch(facecolor=SPICE_DARK, label="Projected (seasonal model)"),
]
axes[0].legend(handles=legend_elements, fontsize=9)

# Right: Annual revenue under three scenarios
annual_revenues = {}
for name, rate in scenarios.items():
    annual_revenues[name] = annual_total * rate

rev_names = list(annual_revenues.keys())
rev_values = list(annual_revenues.values())

bars = axes[1].barh(rev_names, rev_values, color=colors_scenario, edgecolor="white", height=0.5)
axes[1].set_xlabel("Estimated Annual Revenue (CAD)")
axes[1].set_title("Annual Revenue Scenarios — Bissell 30 kWp",
                   fontweight="bold", color=SPICE_DARK)

for i, v in enumerate(rev_values):
    axes[1].text(v + 10, i, f"${v:,.0f}", va="center", fontweight="bold")

plt.tight_layout()
plt.show()

print(f"\nAnnual Revenue Scenarios (projected {annual_total:,.0f} kWh):")
for name, rev in annual_revenues.items():
    print(f"  {name}: ${rev:,.2f}")

---\n# Section 7: Portfolio-Wide Forward Planning\n\nScale the Bissell analysis to SPICE's full **88.62 kWp portfolio** (4 projects) to provide a strategic\noutlook for the cooperative's board and investors.

In [ ]:
# --- Portfolio-Wide Strategic Projections ---

PORTFOLIO_KWP = 88.62  # Total across all 4 SPICE projects
EDMONTON_YIELD = 1100   # kWh/kWp/year (NRCan PV potential for Edmonton)

portfolio_annual_kwh = PORTFOLIO_KWP * EDMONTON_YIELD
portfolio_co2_tonnes = portfolio_annual_kwh / 1000 * 0.45  # Alberta emission factor

print("SPICE PORTFOLIO — STRATEGIC OUTLOOK")
print("=" * 65)
print(f"Total installed capacity: {PORTFOLIO_KWP} kWp across 4 projects")
print(f"Estimated annual production: {portfolio_annual_kwh:,.0f} kWh")
print(f"Estimated annual CO2 avoided: {portfolio_co2_tonnes:,.1f} tonnes")
print()

# Revenue scenarios for full portfolio
print(f"{'Scenario':<35s} {'Annual Revenue':>15s} {'5-Year':>12s} {'20-Year':>12s}")
print("-" * 75)
for name, rate in scenarios.items():
    annual = portfolio_annual_kwh * rate
    five_yr = annual * 5
    twenty_yr = annual * 20
    print(f"{name:<35s} ${annual:>12,.0f}  ${five_yr:>10,.0f}  ${twenty_yr:>10,.0f}")

print("-" * 75)
print(f"\nNote: These projections assume constant rates. In practice, electricity")
print(f"prices increase ~2-4% annually, so actual 20-year revenue will be higher.")

In [ ]:
# --- Portfolio visualization: 20-year outlook ---

years = np.arange(1, 21)
rate_increase = 0.03  # 3% annual electricity price escalation

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Cumulative revenue over 20 years
for (name, rate), color in zip(scenarios.items(), colors_scenario):
    cum_rev = np.cumsum([portfolio_annual_kwh * rate * (1 + rate_increase) ** y for y in range(20)])
    axes[0].plot(years, cum_rev / 1000, color=color, linewidth=2.5, marker="o", markersize=4, label=name)

axes[0].set_title("SPICE Portfolio — 20-Year Cumulative Revenue\n(with 3% annual price escalation)",
                   fontweight="bold", color=SPICE_DARK)
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Cumulative Revenue ($K CAD)")
axes[0].legend(fontsize=9)

# Right: Cumulative environmental impact
cum_co2 = portfolio_co2_tonnes * years
cum_trees = cum_co2 / 0.022

ax_r = axes[1]
ax_r2 = ax_r.twinx()

ax_r.bar(years, cum_co2, color=SPICE_GREEN, alpha=0.7, label="Cumulative CO₂ Avoided (tonnes)")
ax_r2.plot(years, cum_trees, color=SPICE_DARK, linewidth=2.5, marker="s", markersize=4,
           label="Equivalent Trees")

ax_r.set_title("SPICE Portfolio — 20-Year Environmental Impact",
               fontweight="bold", color=SPICE_DARK)
ax_r.set_xlabel("Year")
ax_r.set_ylabel("Cumulative CO₂ Avoided (tonnes)", color=SPICE_GREEN)
ax_r2.set_ylabel("Equivalent Trees Planted", color=SPICE_DARK)

lines1, labels1 = ax_r.get_legend_handles_labels()
lines2, labels2 = ax_r2.get_legend_handles_labels()
ax_r.legend(lines1 + lines2, labels1 + labels2, loc="upper left", fontsize=9)

plt.tight_layout()
plt.show()

---\n# Section 8: Summary & Recommendations\n\n## Key Findings\n\n### Seasonal Decomposition\n- **Trend:** Clear downward trajectory from ~136 kWh/day (August) to near-zero (December), driven by Edmonton's seasonal solar decline.\n- **Weekly seasonality:** Mild 7-day fluctuation (~10-15 kWh amplitude), likely reflecting weekly weather cycles.\n- **Residuals:** Day-to-day weather variability accounts for ~20 kWh std deviation.\n\n### 30-Day Production Forecast\n- Holt-Winters model forecasts very low December production, consistent with Edmonton receiving only 0.9 kWh/m²/day of solar irradiance in winter.\n- **95% confidence interval** captures the range of uncertainty around the forecast.\n- The model correctly captures the seasonal shutdown behavior.\n\n### Revenue Scenarios\n| Scenario | 30-Day (Dec) | Annual (Bissell) | Annual (Portfolio) |\n|---|---|---|---|\n| Best Case (RRO $0.18/kWh) | See output | See output | See output |\n| Base Case (Micro-gen $0.12/kWh) | See output | See output | See output |\n| Worst Case (AESO Pool) | See output | See output | See output |\n\n### External Data Integration\n- **AESO prices:** Used to build the worst-case revenue scenario reflecting wholesale market conditions.\n- **Weather (Edmonton solar profile):** Used to explain the seasonal production decline and calibrate annual projections.\n\n## Recommendations for SPICE Stakeholders\n\n1. **Winter months contribute minimal production** — Budget and communicate this seasonal reality to members.\n2. **Revenue sensitivity:** The gap between best and worst case pricing is significant — securing a favorable PPA or micro-gen credit rate has high financial impact.\n3. **Portfolio diversification matters:** Scaling to all 4 projects (88.62 kWp) significantly improves total revenue and environmental impact.\n4. **Forward planning:** Use the 20-year projections with 3% price escalation for investment case presentations.\n\n## Data Sources & References\n- **Production data:** Fronius Solar.web export — Bissell Thrift Store (provided by Annette Dautel, SPICE)\n- **AESO prices:** Alberta Electric System Operator monthly pool price reports (aeso.ca)\n- **Edmonton solar resource:** Natural Resources Canada PV potential data\n- **Alberta emission factor:** 0.45 tonnes CO₂e/MWh (Government of Alberta TIER regulation, 2023)\n- **Alberta electricity rate:** ~$0.18/kWh blended RRO (Alberta Utilities Commission)